# 99 - Reset do ambiente demo

Este notebook está reservado para reset/limpeza do ambiente. Adicione as células necessárias conforme o roteiro da demo.

In [1]:
import requests
import time
import json
import urllib3

urllib3.disable_warnings()

# =====================================================================
# CONFIGURAÇÕES LIVY3
# =====================================================================

LIVY_URL = "<datahub_link>"
USERNAME = "<usuário>"
PASSWORD = "<senha>"

# =====================================================================
# CONFIGURAÇÕES SPARK
# =====================================================================

SESSION_CONF = {
    "spark.app.name": "PRODEMGE_Iceberg_Reset",
    "spark.executor.memory": "4g",
    "spark.executor.cores": "2",
    "spark.executor.instances": "2",
    "spark.driver.memory": "2g",

    "spark.sql.catalog.iceberg_prod": "org.apache.iceberg.spark.SparkCatalog",
    "spark.sql.catalog.iceberg_prod.type": "hive",
    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
}

# =====================================================================
# CLIENTE HTTP
# =====================================================================

http = requests.Session()
http.auth = (USERNAME, PASSWORD)
http.verify = False

HEADERS = {
    "Content-Type": "application/json"
}


# =====================================================================
# FUNÇÕES AUXILIARES
# =====================================================================

def create_session():

    payload = {
        "kind": "pyspark",
        "conf": SESSION_CONF
    }

    response = http.post(
        f"{LIVY_URL}/sessions",
        headers=HEADERS,
        data=json.dumps(payload)
    )

    response.raise_for_status()

    session_id = response.json()["id"]

    print(f"Sessão criada: {session_id}")

    while True:

        response = http.get(
            f"{LIVY_URL}/sessions/{session_id}"
        )

        response.raise_for_status()

        state = response.json()["state"]

        print(f"Estado da sessão: {state}")

        if state == "idle":
            print("Sessão pronta.")
            return session_id

        if state in ["dead", "error", "killed"]:
            raise RuntimeError("Falha ao iniciar sessão Livy.")

        time.sleep(5)


def submit_statement(session_id, code):

    payload = {
        "code": code,
        "kind": "pyspark"
    }

    response = http.post(
        f"{LIVY_URL}/sessions/{session_id}/statements",
        headers=HEADERS,
        data=json.dumps(payload)
    )

    response.raise_for_status()

    statement_id = response.json()["id"]

    print(f"Statement enviado: {statement_id}")

    while True:

        response = http.get(
            f"{LIVY_URL}/sessions/{session_id}/statements/{statement_id}"
        )

        response.raise_for_status()

        result = response.json()

        state = result["state"]

        print(f"Estado do statement: {state}")

        if state == "available":

            output = result.get("output", {})

            if output.get("status") == "ok":

                data = output.get("data", {})

                if "text/plain" in data:
                    print(data["text/plain"])

            else:
                print(json.dumps(output, indent=2, ensure_ascii=False))

            return

        if state in ["error", "cancelled", "cancelling"]:

            print(json.dumps(result, indent=2, ensure_ascii=False))

            raise RuntimeError("Erro ao executar statement.")

        time.sleep(3)


def close_session(session_id):

    response = http.delete(
        f"{LIVY_URL}/sessions/{session_id}"
    )

    print(f"Sessão encerrada: {session_id}")


# =====================================================================
# CRIAÇÃO DA SESSÃO
# =====================================================================

livy_session_id = create_session()

# =====================================================================
# RESET COMPLETO DO AMBIENTE
# =====================================================================

reset_code = """
print("=" * 80)
print("RESET DO AMBIENTE ICEBERG PRODEMGE")
print("=" * 80)

# =====================================================================
# DATABASE
# =====================================================================

spark.sql(\"\"\"
CREATE DATABASE IF NOT EXISTS governo_mg
\"\"\")

# =====================================================================
# DROP DAS TABELAS DA DEMO
# =====================================================================

tables = [

    "beneficios",
    "beneficios_particionados",
    "beneficios_staging"

]

for table_name in tables:

    full_table_name = f"governo_mg.{table_name}"

    print(f"Removendo tabela: {full_table_name}")

    spark.sql(f\"\"\"
    DROP TABLE IF EXISTS {full_table_name}
    \"\"\")

# =====================================================================
# LIMPEZA DE METADADOS
# =====================================================================

print("\\n")
print("=" * 80)
print("VALIDANDO TABELAS RESTANTES")
print("=" * 80)

spark.sql(\"\"\"
SHOW TABLES IN governo_mg
\"\"\").show(truncate=False)

print("\\n")
print("=" * 80)
print("RESET CONCLUÍDO")
print("=" * 80)

print("Agora a demo pode ser executada novamente do notebook 00 até o 10.")
"""

submit_statement(
    livy_session_id,
    reset_code
)

# =====================================================================
# ENCERRAMENTO DA SESSÃO
# =====================================================================

close_session(livy_session_id)

Sessão criada: 14
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: starting
Estado da sessão: idle
Sessão pronta.
Statement enviado: 0
Estado do statement: waiting
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: running
Estado do statement: available
RESET DO AMBIENTE ICEBERG PRODEMGE
Removendo tabela: governo_mg.beneficios
Removendo tabela: governo_mg.beneficios_particionados
Removendo tabela: governo_mg.beneficios_staging


VALIDANDO TABELAS RESTANTES
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



RESET CON